In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# ============================================
# 1. SIMPLE RNN MODEL
# ============================================
class SimpleRNNModel(nn.Module):
    """
    Simple RNN model for sequence classification
    """
    def __init__(self, input_size, hidden_size=64, num_layers=1, num_classes=2, dropout=0.2):
        super(SimpleRNNModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            nonlinearity='tanh'
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.fc2 = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Initialize hidden state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # Forward propagate RNN
        out, _ = self.rnn(x, h0)

        # Take the last time step
        out = out[:, -1, :]

        # Decode
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)

        return out

# ============================================
# 2. LSTM MODEL
# ============================================
class LSTMModel(nn.Module):
    """
    LSTM model for sequence classification
    """
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Initialize hidden and cell states
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # Forward propagate LSTM
        out, _ = self.lstm(x, (h0, c0))

        # Take the last time step
        out = out[:, -1, :]

        # Decode
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.relu(self.fc2(out))
        out = self.fc3(out)

        return out

# ============================================
# 3. TRANSFORMER MODEL
# ============================================
class PositionalEncoding(nn.Module):
    """
    Positional encoding for transformer
    """
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)

        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

class TransformerModel(nn.Module):
    """
    Simple Transformer model for sequence classification
    """
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, num_classes=2, dropout=0.1, max_len=5000):
        super(TransformerModel, self).__init__()

        self.d_model = d_model
        self.input_projection = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)

        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(d_model, 32)
        self.fc2 = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Project input to d_model dimensions
        x = self.input_projection(x)

        # Add positional encoding
        x = self.pos_encoder(x)

        # Apply transformer encoder
        x = self.transformer_encoder(x)

        # Global average pooling over sequence dimension
        x = x.mean(dim=1)

        # Classification head
        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

# ============================================
# SIMPLIFIED TRANSFORMER WITH CUSTOM ATTENTION
# ============================================
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism
    """
    def __init__(self, d_model, nhead, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead

        assert self.head_dim * nhead == d_model, "d_model must be divisible by nhead"

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # Linear projections and reshape
        Q = self.query(query).view(batch_size, -1, self.nhead, self.head_dim).transpose(1, 2)
        K = self.key(key).view(batch_size, -1, self.nhead, self.head_dim).transpose(1, 2)
        V = self.value(value).view(batch_size, -1, self.nhead, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        # Apply attention to values
        out = torch.matmul(attention_weights, V)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        out = self.out_proj(out)

        return out

class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network
    """
    def __init__(self, d_model, dim_feedforward, dropout=0.1):
        super(FeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, dim_feedforward)
        self.fc2 = nn.Linear(dim_feedforward, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.dropout(self.relu(self.fc1(x))))

class TransformerBlock(nn.Module):
    """
    Single Transformer block with multi-head attention and feed-forward network
    """
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super(TransformerBlock, self).__init__()
        self.attention = MultiHeadAttention(d_model, nhead, dropout)
        self.feed_forward = FeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_output = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        # Feed-forward with residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

class CustomTransformerModel(nn.Module):
    """
    Custom Transformer implementation for learning purposes
    """
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, num_classes=2, dropout=0.1, max_len=5000):
        super(CustomTransformerModel, self).__init__()

        self.d_model = d_model
        self.input_projection = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)

        # Stack transformer blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(d_model, 32)
        self.fc2 = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Project input
        x = self.input_projection(x)

        # Add positional encoding
        x = self.pos_encoder(x)

        # Apply transformer blocks
        for block in self.transformer_blocks:
            x = block(x)

        # Global average pooling
        x = x.mean(dim=1)

        # Classification head
        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

# ============================================
# TRAINING UTILITIES
# ============================================
def train_model(model, train_loader, val_loader, epochs=10, lr=0.001, device='cuda'):
    """
    Generic training function for PyTorch models
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation phase
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                _, predicted = torch.max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()

        val_accuracy = 100 * correct / total
        val_accuracies.append(val_accuracy)

        if (epoch + 1) % 5 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_train_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%')

    return train_losses, val_accuracies

def evaluate_model(model, test_loader, device='cuda'):
    """
    Evaluate model on test set
    """
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

    accuracy = 100 * correct / total
    return accuracy

# ============================================
# DEMONSTRATION WITH SYNTHETIC DATA
# ============================================
def generate_synthetic_data(n_samples=1000, seq_length=50, n_features=1):
    """
    Generate synthetic sequential data for classification
    """
    X = np.random.randn(n_samples, seq_length, n_features).astype(np.float32)
    # Simple classification based on mean value
    y = (X.mean(axis=(1, 2)) > 0).astype(np.int64)
    return X, y

# Generate data
seq_length = 50
n_features = 1
X, y = generate_synthetic_data(n_samples=1000, seq_length=seq_length, n_features=n_features)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Split training for validation
val_size = int(0.2 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader_subset = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}\n")

# ============================================
# TRAIN AND EVALUATE MODELS
# ============================================

# 1. RNN Model
print("="*50)
print("TRAINING RNN MODEL")
print("="*50)
rnn_model = SimpleRNNModel(
    input_size=n_features,
    hidden_size=64,
    num_layers=1,
    num_classes=2,
    dropout=0.2
)
print(f"RNN Model parameters: {sum(p.numel() for p in rnn_model.parameters()):,}")
rnn_train_losses, rnn_val_acc = train_model(
    rnn_model, train_loader_subset, val_loader,
    epochs=10, lr=0.001, device=device
)
rnn_test_acc = evaluate_model(rnn_model, test_loader, device=device)
print(f"RNN Test Accuracy: {rnn_test_acc:.2f}%\n")

# 2. LSTM Model
print("="*50)
print("TRAINING LSTM MODEL")
print("="*50)
lstm_model = LSTMModel(
    input_size=n_features,
    hidden_size=128,
    num_layers=2,
    num_classes=2,
    dropout=0.2
)
print(f"LSTM Model parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")
lstm_train_losses, lstm_val_acc = train_model(
    lstm_model, train_loader_subset, val_loader,
    epochs=10, lr=0.001, device=device
)
lstm_test_acc = evaluate_model(lstm_model, test_loader, device=device)
print(f"LSTM Test Accuracy: {lstm_test_acc:.2f}%\n")

# 3. Transformer Model (PyTorch's implementation)
print("="*50)
print("TRAINING TRANSFORMER MODEL (PyTorch)")
print("="*50)
transformer_model = TransformerModel(
    input_size=n_features,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    num_classes=2,
    dropout=0.1
)
print(f"Transformer Model parameters: {sum(p.numel() for p in transformer_model.parameters()):,}")
transformer_train_losses, transformer_val_acc = train_model(
    transformer_model, train_loader_subset, val_loader,
    epochs=10, lr=0.001, device=device
)
transformer_test_acc = evaluate_model(transformer_model, test_loader, device=device)
print(f"Transformer Test Accuracy: {transformer_test_acc:.2f}%\n")

# 4. Custom Transformer Model (for learning)
print("="*50)
print("TRAINING CUSTOM TRANSFORMER MODEL")
print("="*50)
custom_transformer = CustomTransformerModel(
    input_size=n_features,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    num_classes=2,
    dropout=0.1
)
print(f"Custom Transformer parameters: {sum(p.numel() for p in custom_transformer.parameters()):,}")
custom_train_losses, custom_val_acc = train_model(
    custom_transformer, train_loader_subset, val_loader,
    epochs=10, lr=0.001, device=device
)
custom_test_acc = evaluate_model(custom_transformer, test_loader, device=device)
print(f"Custom Transformer Test Accuracy: {custom_test_acc:.2f}%\n")

# ============================================
# TEXT CLASSIFICATION EXAMPLE (with embeddings)
# ============================================
class RNNTextClassifier(nn.Module):
    """
    RNN for text classification with embeddings
    """
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=64, num_classes=2, dropout=0.2):
        super(RNNTextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        output = output[:, -1, :]  # Take last time step
        output = self.dropout(output)
        return self.fc(output)

class LSTMTextClassifier(nn.Module):
    """
    LSTM for text classification with embeddings
    """
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=128, num_layers=2, num_classes=2, dropout=0.2):
        super(LSTMTextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        output = output[:, -1, :]  # Take last time step
        output = self.dropout(output)
        return self.fc(output)

class TransformerTextClassifier(nn.Module):
    """
    Transformer for text classification with embeddings
    """
    def __init__(self, vocab_size, embedding_dim=128, nhead=4, num_layers=2,
                 dim_feedforward=256, num_classes=2, dropout=0.1, max_len=5000):
        super(TransformerTextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pos_encoder = PositionalEncoding(embedding_dim, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.pos_encoder(embedded)
        output = self.transformer(embedded)
        output = output.mean(dim=1)  # Global average pooling
        output = self.dropout(output)
        return self.fc(output)

print("="*50)
print("TEXT CLASSIFICATION MODELS SUMMARY")
print("="*50)

# Example text model configurations
vocab_size = 10000
max_length = 100

rnn_text = RNNTextClassifier(vocab_size=vocab_size)
lstm_text = LSTMTextClassifier(vocab_size=vocab_size)
transformer_text = TransformerTextClassifier(vocab_size=vocab_size)

print(f"RNN Text Model: {sum(p.numel() for p in rnn_text.parameters()):,} parameters")
print(f"LSTM Text Model: {sum(p.numel() for p in lstm_text.parameters()):,} parameters")
print(f"Transformer Text Model: {sum(p.numel() for p in transformer_text.parameters()):,} parameters")

# Function to save/load models
def save_model(model, filepath):
    torch.save(model.state_dict(), filepath)
    print(f"Model saved to {filepath}")

def load_model(model, filepath):
    model.load_state_dict(torch.load(filepath))
    model.eval()
    print(f"Model loaded from {filepath}")
    return model

Using device: cpu
Training data shape: (800, 50, 1)
Test data shape: (200, 50, 1)

TRAINING RNN MODEL
RNN Model parameters: 6,434
Epoch [5/10], Loss: 0.6917, Val Accuracy: 45.00%
Epoch [10/10], Loss: 0.6238, Val Accuracy: 78.75%
RNN Test Accuracy: 76.00%

TRAINING LSTM MODEL
LSTM Model parameters: 209,570
Epoch [5/10], Loss: 0.2543, Val Accuracy: 88.75%
Epoch [10/10], Loss: 0.1711, Val Accuracy: 95.00%
LSTM Test Accuracy: 96.00%

TRAINING TRANSFORMER MODEL (PyTorch)
Transformer Model parameters: 69,218
Epoch [5/10], Loss: 0.2227, Val Accuracy: 88.75%
Epoch [10/10], Loss: 0.1836, Val Accuracy: 83.75%
Transformer Test Accuracy: 86.00%

TRAINING CUSTOM TRANSFORMER MODEL
Custom Transformer parameters: 69,218
Epoch [5/10], Loss: 0.1737, Val Accuracy: 95.62%
Epoch [10/10], Loss: 0.1580, Val Accuracy: 88.12%
Custom Transformer Test Accuracy: 86.00%

TEXT CLASSIFICATION MODELS SUMMARY
RNN Text Model: 1,292,546 parameters
LSTM Text Model: 1,544,450 parameters
Transformer Text Model: 1,545,218 p

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:641: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__(mode, *args, **kwargs)
